<a href="https://colab.research.google.com/github/Varada-Kachroo/HimalayanGuardian/blob/main/himalayan_guardian_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=180000)

import os
base = '/content/drive/MyDrive/HimalayanGuardian'
os.makedirs(f'{base}/raw', exist_ok=True)
os.makedirs(f'{base}/processed', exist_ok=True)
print("Folders ready:", os.listdir(base))

Mounted at /content/drive
Folders ready: ['raw', 'processed', 'south_lhonak_s2_rgb.tif']


In [ ]:
!pip install earthengine-api geopandas rasterio requests -q

import ee
ee.Authenticate()
ee.Initialize(project='southern-frame-509805-f2')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
!pip install --force-reinstall --no-deps fiona -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.4 MB/s eta 0:00:00


In [ ]:
# Approximate coordinates for South Lhonak Lake, Sikkim
point = ee.Geometry.Point([88.18, 27.9])
region = point.buffer(5000)  # 5 km radius

# Sentinel-2 (surface reflectance), recent cloud-free image
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(region)
      .filterDate('2023-01-01', '2023-12-31')
      .sort('CLOUDY_PIXEL_PERCENTAGE')
      .first())

# SRTM elevation
srtm = ee.Image('USGS/SRTMGL1_003').clip(region)

print("Sentinel-2 image ID:", s2.get('system:index').getInfo())
print("SRTM band names:", srtm.bandNames().getInfo())

Sentinel-2 image ID: 20230114T044141_20230114T045036_T45RXM
SRTM band names: ['elevation']


In [ ]:
!pip install geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.3 MB/s eta 0:00:00


In [ ]:
import geemap

Map = geemap.Map(basemap='HYBRID')  # use Google satellite basemap instead of OSM
Map.centerObject(region, 12)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 RGB')
Map.addLayer(srtm, {'min': 0, 'max': 6000, 'palette': ['blue', 'green', 'yellow', 'red']}, 'Elevation')
Map

Map(center=[27.900005129440906, 88.18000016095561], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import requests

url = "https://stacks.stanford.edu/object/nf183jx2059"
response = requests.get(url)

with open('/content/rgi_southeast_asia.zip', 'wb') as f:
    f.write(response.content)

print("Downloaded:", len(response.content), "bytes")

Downloaded: 15576391 bytes


In [ ]:
import zipfile

with zipfile.ZipFile('/content/rgi_southeast_asia.zip', 'r') as z:
    z.extractall('/content/rgi_data')

import os
print(os.listdir('/content/rgi_data'))

['15_rgi50_SouthAsiaEast.shx', '15_rgi50_SouthAsiaEast.shp.xml', '15_rgi50_SouthAsiaEast.shp', '15_rgi50_SouthAsiaEast.prj', '15_rgi50_SouthAsiaEast-iso19139.xml', '15_rgi50_SouthAsiaEast-iso19110.xml', '15_rgi50_SouthAsiaEast.dbf']


In [ ]:
import geopandas as gpd

glaciers = gpd.read_file('/content/rgi_data/15_rgi50_SouthAsiaEast.shp')
print("Total glaciers in region:", len(glaciers))
print(glaciers.columns.tolist())

# Filter to glaciers near South Lhonak Lake (within ~0.5 degree box)
lhonak_area = glaciers.cx[87.9:88.5, 27.6:28.2]
print("Glaciers near South Lhonak:", len(lhonak_area))
lhonak_area.head()

Total glaciers in region: 13119
['geometry']
Glaciers near South Lhonak: 533


,geometry
2696,"POLYGON ((88.18081 27.89307, 88.18085 27.89306..."
2697,"MULTIPOLYGON (((88.1582 27.8835, 88.15773 27.8..."
2698,"POLYGON ((88.13868 27.87819, 88.13883 27.87803..."
2699,"POLYGON ((88.18136 27.89887, 88.18111 27.89933..."
2700,"POLYGON ((88.22449 27.86925, 88.22453 27.86925..."


In [ ]:
# Reproject to a metric CRS to get accurate area in km²
lhonak_glaciers = glaciers.cx[87.9:88.5, 27.6:28.2].copy()
lhonak_glaciers_metric = lhonak_glaciers.to_crs(epsg=32645)  # UTM zone for Sikkim
lhonak_glaciers['area_km2'] = lhonak_glaciers_metric.area / 1e6

print(lhonak_glaciers[['area_km2']].describe())
lhonak_glaciers.head()

         area_km2
count  533.000000
mean     1.552762
std      5.177434
min      0.011736
25%      0.154164
50%      0.381570
75%      1.014867
max     76.087358


,geometry,area_km2
2696,"POLYGON ((88.18081 27.89307, 88.18085 27.89306...",0.526042
2697,"MULTIPOLYGON (((88.1582 27.8835, 88.15773 27.8...",12.577403
2698,"POLYGON ((88.13868 27.87819, 88.13883 27.87803...",3.009360
2699,"POLYGON ((88.18136 27.89887, 88.18111 27.89933...",0.312573
2700,"POLYGON ((88.22449 27.86925, 88.22453 27.86925...",0.081430


In [ ]:
import requests
import pandas as pd

lat, lon = 27.9, 88.18  # South Lhonak Lake

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M",  # rainfall (mm/day), temperature at 2m (°C)
    "community": "AG",
    "longitude": lon,
    "latitude": lat,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

weather_df = pd.DataFrame(data['properties']['parameter'])
weather_df.index = pd.to_datetime(weather_df.index, format='%Y%m%d')
print(weather_df.head())
print(weather_df.describe())

            PRECTOTCORR    T2M
2023-01-01          0.0  -9.54
2023-01-02          0.0 -10.94
2023-01-03          0.0 -11.36
2023-01-04          0.0 -12.26
2023-01-05          0.0 -12.51
       PRECTOTCORR         T2M
count   365.000000  365.000000
mean      2.766548   -1.397753
std       4.009117    6.157279
min       0.000000  -13.410000
25%       0.000000   -6.850000
50%       0.650000   -2.500000
75%       4.410000    4.640000
max      19.540000    8.810000


In [ ]:
print("Status code:", response.status_code)
print("Response text:", response.text[:500])

Status code: 200
Response text: {"type":"Feature","geometry":{"type":"Point","coordinates":[88.18,27.9,4792.39]},"properties":{"parameter":{"PRECTOTCORR":{"20230101":0.0,"20230102":0.0,"20230103":0.0,"20230104":0.0,"20230105":0.0,"20230106":0.0,"20230107":0.0,"20230108":0.0,"20230109":0.0,"20230110":0.0,"20230111":0.0,"20230112":0.0,"20230113":0.0,"20230114":0.04,"20230115":0.1,"20230116":0.0,"20230117":0.0,"20230118":0.02,"20230119":0.1,"20230120":0.0,"20230121":0.0,"20230122":0.0,"20230123":0.0,"20230124":0.0,"20230125":0.0,


In [ ]:
import requests, json, pandas as pd

query = """
[out:json][timeout:90];
node["place"~"village|town|hamlet"](27.2,88.1,27.95,88.85);
out;
"""

osm_valley = None
for url in ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]:
    try:
        r = requests.post(url, data={'data': query}, headers=headers, timeout=120)
        print(url, "->", r.status_code)
        if r.status_code == 200:
            osm_valley = r.json()
            break
    except Exception as e:
        print(url, "-> failed:", e)

if osm_valley:
    with open('/content/drive/MyDrive/HimalayanGuardian/raw/osm_valley.json', 'w') as f:
        json.dump(osm_valley, f)
    rows = [{'name': el.get('tags', {}).get('name:en') or el.get('tags', {}).get('name', 'unnamed'),
             'lat': el['lat'], 'lon': el['lon'], 'place': el.get('tags', {}).get('place')}
            for el in osm_valley['elements']]
    valley_df = pd.DataFrame(rows).sort_values('lat', ascending=False)
    print(len(valley_df), "places found")
    print(valley_df.to_string())

https://overpass-api.de/api/interpreter -> failed: name 'headers' is not defined
https://overpass.kumi.systems/api/interpreter -> failed: name 'headers' is not defined


In [ ]:
import requests, json, time

headers = {'User-Agent': 'HimalayanGuardianMiniproject/1.0'}
river_query = """
[out:json][timeout:60];
way["waterway"="river"]["name"~"Teesta|Tista|Chu|Lachen"](27.2,88.1,27.95,88.85);
out tags center;
"""

osm_rivers = None
for attempt in range(2):
    for url in ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]:
        try:
            r = requests.post(url, data={'data': river_query}, headers=headers, timeout=90)
            print(url, "->", r.status_code)
            if r.status_code == 200:
                osm_rivers = r.json()
                break
        except Exception as e:
            print(url, "-> failed:", e)
    if osm_rivers:
        break
    time.sleep(20)

if osm_rivers:
    with open('/content/drive/MyDrive/HimalayanGuardian/raw/osm_rivers.json', 'w') as f:
        json.dump(osm_rivers, f)
    print(len(osm_rivers['elements']), "river segments")
    print(sorted({el.get('tags', {}).get('name', 'unnamed') for el in osm_rivers['elements']}))

https://overpass-api.de/api/interpreter -> 200
48 river segments
['Chakung Glong Chu', 'Di Chu', 'Dik Chu', 'Dre Chu', 'Goma Chu', 'Kaheng Chu', 'Kalet Chu', 'Lachen Chu', 'Lachung Chu', 'Lasha Chu', 'Lhonak Chu', 'Prek Chu', 'Rangpo Chu', 'Rangrang Chu', 'Rathong Chu', 'Rimbi Chu', 'Ringi Chu', 'Rongbe Chu', 'Rongli Chu', 'Roro Chu', 'Sebozung Chu', 'Talung Chu', 'Teesta', 'Zemu Chu']


In [ ]:
import requests, json, time

headers = {'User-Agent': 'HimalayanGuardianMiniproject/1.0'}
flood_query = """
[out:json][timeout:60];
way["waterway"="river"]["name"~"^(Lhonak Chu|Zemu Chu|Lachen Chu|Teesta)$"](27.2,88.1,27.95,88.85);
out geom;
"""

flood_rivers = None
for attempt in range(2):
    for url in ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]:
        try:
            r = requests.post(url, data={'data': flood_query}, headers=headers, timeout=90)
            print(url, "->", r.status_code)
            if r.status_code == 200:
                flood_rivers = r.json()
                break
        except Exception as e:
            print(url, "-> failed:", e)
    if flood_rivers:
        break
    time.sleep(20)

if flood_rivers:
    with open('/content/drive/MyDrive/HimalayanGuardian/raw/flood_path_rivers.json', 'w') as f:
        json.dump(flood_rivers, f)
    print(len(flood_rivers['elements']), "river segments saved")

https://overpass-api.de/api/interpreter -> 200
15 river segments saved


In [ ]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point

base = '/content/drive/MyDrive/HimalayanGuardian'

with open(f'{base}/raw/flood_path_rivers.json') as f:
    rivers = json.load(f)
with open(f'{base}/raw/osm_valley.json') as f:
    valley = json.load(f)

# Rivers -> lines, in metric CRS
lines = [LineString([(p['lon'], p['lat']) for p in el['geometry']])
         for el in rivers['elements'] if 'geometry' in el]
rivers_m = gpd.GeoSeries(lines, crs=4326).to_crs(32645)
flood_zone = rivers_m.buffer(2000).union_all()   # 2 km either side of the river

# Villages -> points, in metric CRS
rows = [{'name': el.get('tags', {}).get('name:en') or el.get('tags', {}).get('name', 'unnamed'),
         'place': el.get('tags', {}).get('place'),
         'lat': el['lat'], 'lon': el['lon']} for el in valley['elements']]
v = gpd.GeoDataFrame(rows, geometry=[Point(r['lon'], r['lat']) for r in rows], crs=4326).to_crs(32645)

lake = gpd.GeoSeries([Point(88.18, 27.9)], crs=4326).to_crs(32645).iloc[0]
v['dist_to_river_km'] = v.geometry.apply(lambda p: rivers_m.distance(p).min()) / 1000
v['dist_from_lake_km'] = v.geometry.distance(lake) / 1000

flood_path = v[(v.geometry.within(flood_zone)) & (v['name'] != 'unnamed')]
flood_path = flood_path.sort_values('dist_from_lake_km')[['name', 'place', 'dist_to_river_km', 'dist_from_lake_km']]
flood_path = flood_path.round(1)

print(len(flood_path), "villages along the flood path")
print(flood_path.to_string())

flood_path.to_csv(f'{base}/processed/flood_path_villages.csv', index=False)

22 villages along the flood path
                           name    place  dist_to_river_km  dist_from_lake_km
80                Thangu Valley  village               0.2               35.1
75                       Bamzey   hamlet               1.2               35.9
13                       Lachen     town               0.5               41.0
60                  Passingdong  village               2.0               52.8
23                   Sangkalang  village               0.3               55.5
49                       Sentam  village               1.1               55.9
0                        Mangan     town               0.9               56.4
86                   Chungthang  village               0.2               56.5
72                        Sudur  village               0.9               57.5
73                    Rapchyong  village               1.0               57.6
74                    Bringbong  village               0.3               57.7
69  Gnon-Sangdong, Lower Dzongu

In [ ]:
import requests, zipfile, os

url = "https://zenodo.org/records/18257243/files/fidelsteiner/HMAGLOFDB-v1.3.0.zip?download=1"
r = requests.get(url, timeout=120)
print("Status:", r.status_code, "| bytes:", len(r.content))

zip_path = '/content/HMAGLOFDB.zip'
with open(zip_path, 'wb') as f:
    f.write(r.content)

with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/HMAGLOFDB')

for root, dirs, files in os.walk('/content/HMAGLOFDB'):
    for name in files:
        print(os.path.join(root, name))

Status: 200 | bytes: 107879
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/LICENSE
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/HMAGLOFDB_Metadata.yml
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/README.md
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/gitLOGO.png
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/Database/GLOFs/HMAGLOFDB_removed.csv
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/Database/GLOFs/HMAGLOFDB.csv
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/Database/GLOFs/schema.ini
/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/Code/HMAGLOFDB_PubAnalysis.R


In [ ]:
import pandas as pd

path = '/content/HMAGLOFDB/fidelsteiner-HMAGLOFDB-1d975de/Database/GLOFs/HMAGLOFDB.csv'
glof = pd.read_csv(path, encoding='latin1')

print("Shape:", glof.shape)
print(glof.columns.tolist())
print(glof.head(3).T)

Shape: (768, 59)
['GF_ID', 'Year_approx', 'Year_exact', 'Month', 'Day', 'Lake_name', 'Glacier_name', 'GL_ID', 'LakeDB_ID', 'G_ID', 'Lat_lake', 'Lon_lake', 'Elev_lake', 'Lat_impact', 'Lon_impact', 'Elev_impact', 'Impact_type', 'Lake_type', 'Transboundary', 'Repeat', 'Region_RGI', 'Region_HiMAP', 'Country', 'Province', 'River_Basin', 'Driver_lake', 'Driver_GLOF', 'Mechanism', 'Area', 'Volume', 'Discharge_water', 'Discharge_solid', 'Impact', 'Lives_total', 'Lives_male', 'Lives_female', 'Lives_disabilities', 'Injured_total', 'Injured_male', 'Injured_female', 'Injured_disabilities', 'Displaced_total', 'Displaced_male', 'Displaced_female', 'Displaced_disabilities', 'Livestock', 'Residential_destroyed', 'Commerical_destroyed', 'Residential_damaged', 'Commerical_damaged', 'Infra', 'Agricultural', 'Hydropower', 'Econ_damage', 'Sat_evidence', 'Ref_scientific', 'Ref_scientific_full', 'Ref_other', 'Remarks']
                                                                        0  \
GF_ID        

In [ ]:
import pandas as pd

# 1. How many distinct lakes actually burst?
has_xy = glof.dropna(subset=['Lat_lake', 'Lon_lake']).copy()
has_xy['lake_key'] = has_xy['Lat_lake'].round(2).astype(str) + ',' + has_xy['Lon_lake'].round(2).astype(str)
print("Events:", len(glof), "| with coordinates:", len(has_xy))
print("Distinct lakes (by coordinates):", has_xy['lake_key'].nunique())

# 2. Events by RGI region for the Himalaya
print(glof['Region_RGI'].value_counts().head(8))

# 3. Sikkim area events
sikkim = glof[(glof['Province'].astype(str).str.contains('Sikkim', case=False)) |
              (glof['Lat_lake'].between(27.0, 28.2) & glof['Lon_lake'].between(88.0, 89.0))]
cols = ['GF_ID', 'Year_exact', 'Lake_name', 'Lat_lake', 'Lon_lake', 'Elev_lake', 'Lake_type', 'Driver_GLOF', 'Lives_total']
print("\nSikkim area events:", len(sikkim))
print(sikkim[cols].to_string())

Events: 768 | with coordinates: 768
Distinct lakes (by coordinates): 381
Region_RGI
13_3    252
14_2    204
15_2    132
15_1     31
13_8     28
15_3     23
13_9     23
14_3     19
Name: count, dtype: int64

Sikkim area events: 18
     GF_ID  Year_exact           Lake_name   Lat_lake   Lon_lake  Elev_lake       Lake_type                        Driver_GLOF Lives_total
81      82      1921.0   Lhonak (Ghunsa 1)  27.829351  88.037620     5100.0    Supraglacial                            Unknown         NaN
118    119      1940.0     Qiongbixiama Co  27.850000  88.915000     4767.0  Moraine dammed                            Unknown         NaN
217    219      1970.0             Unknown  27.654000  88.858000     4660.0         Unknown                      Ice avalanche         NaN
218    220      1970.0       Qunbixiama Co  27.847000  88.921000     4670.0  Moraine dammed                            Unknown         NaN
325    327      1986.0            Shimbuwa  27.585963  88.027140     4435.0

In [ ]:
base = '/content/drive/MyDrive/HimalayanGuardian'

him = glof[glof['Region_RGI'].astype(str).str.startswith('15')].copy()
him['lake_key'] = him['Lat_lake'].round(2).astype(str) + ',' + him['Lon_lake'].round(2).astype(str)

# One row per lake: earliest and latest burst year, number of bursts
lakes_burst = (him.groupby('lake_key')
    .agg(lat=('Lat_lake', 'first'), lon=('Lon_lake', 'first'), elev=('Elev_lake', 'first'),
         lake_type=('Lake_type', 'first'), n_bursts=('GF_ID', 'count'),
         first_burst=('Year_exact', 'min'), last_burst=('Year_exact', 'max'))
    .reset_index())
lakes_burst['label'] = 1

# Hold out South Lhonak as the test case
is_lhonak = (lakes_burst['lat'].between(27.90, 27.93)) & (lakes_burst['lon'].between(88.18, 88.20))
test_lake = lakes_burst[is_lhonak]
train_burst = lakes_burst[~is_lhonak]

print("Himalaya events:", len(him))
print("Distinct burst lakes:", len(lakes_burst), "| training:", len(train_burst), "| test (South Lhonak):", len(test_lake))
print("Burst lakes with a known year:", lakes_burst['last_burst'].notna().sum())
print("Burst lakes since 2015:", (lakes_burst['last_burst'] >= 2015).sum())

lakes_burst.to_csv(f'{base}/processed/himalaya_burst_lakes.csv', index=False)

Himalaya events: 186
Distinct burst lakes: 162 | training: 160 | test (South Lhonak): 2
Burst lakes with a known year: 55
Burst lakes since 2015: 15


In [ ]:
print(test_lake[['lake_key', 'lat', 'lon', 'elev', 'lake_type', 'n_bursts', 'first_burst', 'last_burst']].to_string())

       lake_key      lat      lon    elev       lake_type  n_bursts  first_burst  last_burst
51  27.91,88.19  27.9121  88.1926  5228.0  Moraine dammed         1       2023.0      2023.0
52   27.91,88.2  27.9130  88.1981  5203.0  Moraine dammed         1          NaN         NaN


In [ ]:
import zipfile, os

zip_path = '/content/drive/MyDrive/HimalayanGuardian/raw/'
files = [f for f in os.listdir(zip_path) if f.endswith('.zip')]
print("Zip files found:", files)

zip_name = files[0]  # adjust if multiple zips
with zipfile.ZipFile(zip_path + zip_name, 'r') as z:
    z.extractall('/content/glacial_lakes_hkh')

for root, dirs, fnames in os.walk('/content/glacial_lakes_hkh'):
    for name in fnames:
        print(os.path.join(root, name))

Zip files found: ['The Status of Glacial Lakes in the Hindu Kush Himalaya.zip']
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/Data Download Agreement.pdf
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/metadata/hkh_glacial_lake.xml
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/metadata/hkh_glacial_lake.pdf
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.sbx
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.shp
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.cpg
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.dbf
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.shp.xml
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.shx
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.sbn
/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.prj


In [ ]:
import geopandas as gpd

lakes = gpd.read_file('/content/glacial_lakes_hkh/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.shp')
print("Total lakes in HKH:", len(lakes))
print(lakes.columns.tolist())
lakes.head(3)

Total lakes in HKH: 25614
['Latitude', 'Longitude', 'GLIMS_ID', 'Type', 'M_Basin', 'Basin', 'Sub_basin', 'Area', 'Altitude', 'Image_ID', 'ID', 'Dist_gr', 'Gr_fed', 'geometry']


,Latitude,Longitude,GLIMS_ID,Type,M_Basin,Basin,Sub_basin,Area,Altitude,Image_ID,ID,Dist_gr,Gr_fed,geometry
0,35.510,70.877,GL070877E35510N,E(o),Amudarya,Kokcha,Kokcha,0.011,4634,l72152035_03520060917,1,1170.0,0,"POLYGON Z ((70.8778 35.50944 0, 70.87771 35.50..."
1,35.499,70.876,GL070876E35499N,E(o),Amudarya,Kokcha,Kokcha,0.050,4780,l72152035_03520060917,2,54.0,1,"POLYGON Z ((70.87537 35.50001 0, 70.87549 35.5..."
2,35.795,69.987,GL069987E35795N,M(o),Amudarya,Surkhab,Surkhab,0.023,4511,l72152035_03520060917,3,90.0,1,"POLYGON Z ((69.98736 35.79548 0, 69.9875 35.79..."


In [ ]:
himalaya_lakes = lakes[lakes['Basin'] == 'Tista'].copy()
print("Lakes in Tista basin:", len(himalaya_lakes))
print(himalaya_lakes[['Latitude','Longitude','Area','Altitude','Type']].describe())

Lakes in Tista basin: 401
         Latitude   Longitude        Area     Altitude
count  401.000000  401.000000  401.000000   401.000000
mean    27.789663   88.545060    0.068112  4829.922693
std      0.192361    0.237523    0.162669   429.279113
min     27.308000   88.049000    0.004000  3542.000000
25%     27.688000   88.342000    0.010000  4487.000000
50%     27.851000   88.603000    0.021000  4923.000000
75%     27.947000   88.764000    0.054000  5160.000000
max     28.059000   88.878000    1.694000  5665.000000


In [ ]:
import numpy as np

burst_lats = lakes_burst['lat'].values
burst_lons = lakes_burst['lon'].values

def is_near_burst(row, threshold_deg=0.02):
    dlat = burst_lats - row['Latitude']
    dlon = burst_lons - row['Longitude']
    dists = np.sqrt(dlat**2 + dlon**2)
    return dists.min() < threshold_deg

himalaya_lakes['near_burst'] = himalaya_lakes.apply(is_near_burst, axis=1)
negatives = himalaya_lakes[~himalaya_lakes['near_burst']].copy()
negatives['label'] = 0

print("Negative examples (no known burst):", len(negatives))

out_cols = ['Latitude', 'Longitude', 'Area', 'Altitude', 'Type', 'Basin', 'Sub_basin', 'label']
negatives[out_cols].to_csv('/content/drive/MyDrive/HimalayanGuardian/processed/negative_lakes.csv', index=False)

Negative examples (no known burst): 359
